In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score
import numpy as np

In [2]:
# Constants
RandomSTATE = 1082
TRAIN_SIZE = 0.2

In [3]:
# Load real data
try:
    df = pd.read_csv("../../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv")
except FileNotFoundError:
    print("Error: The file '../../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv' was not found.")
    exit()

In [4]:
# Data Preprocessing
# Clean up column names
df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)

# Drop label and timestamp
drop_cols = ['timestamp', 'label']
X_real = df.drop(columns=drop_cols)
y_real = df['label']

# Encode labels if needed
if y_real.dtype == object or y_real.dtype.name == 'category':
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y_real = le.fit_transform(y_real)

# Split real data for evaluation
X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
    X_real, y_real, train_size=TRAIN_SIZE, random_state=RandomSTATE, stratify=y_real
)

In [5]:
# Load synthetic (fake) data
try:
    fake_df = pd.read_csv("../../../../Datasets/csv/syntheticSpoofing_gan_data.csv")
except FileNotFoundError:
    print("Error: The file 'synthetic_gan_data.csv' was not found.")
    print("Please generate synthetic data using your GAN first.")
    exit()

In [6]:
print(fake_df.head())
print(fake_df.info())

       q_0_      q_1_      q_2_      q_3_  delta_q_reset_0_  delta_q_reset_1_  \
0 -0.370131 -0.010673 -0.071623  0.891121          0.925614         -0.207104   
1 -0.380949  0.130560  0.026184  0.940724          0.884008          0.111867   
2 -0.376037  0.013518 -0.034838  0.920586          1.056134          0.032988   
3 -0.365949  0.031671  0.042441  0.971457          1.107374          0.010950   
4 -0.411839  0.009586  0.107297  0.882897          1.106318          0.117761   

   delta_q_reset_2_  delta_q_reset_3_  quat_reset_counter      lat_x  ...  \
0         -0.243811          0.182450            3.256799  36.204834  ...   
1          0.013869          0.479237            3.651965  36.204845  ...   
2          0.122955         -0.079506            2.985945  36.204820  ...   
3          0.011517         -0.073389            2.961031  36.204826  ...   
4         -0.036234         -0.205509            3.137218  36.204840  ...   

   v_z_valid  xy_reset_counter  z_reset_counter  v

In [7]:
if fake_df['label'].dtype == object or fake_df['label'].dtype.name == 'category':
    y_fake = le.transform(fake_df['label'])
else:
    y_fake = fake_df['label'].values.astype(int)

X_fake = fake_df[X_real.columns]


In [8]:
# Build and train XGBoost classifier on fake data
model = xgb.XGBClassifier(
    objective='binary:logistic',
    random_state=RandomSTATE,
    eval_metric='mae',
)
model.fit(X_fake, y_fake)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mae', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [9]:
# Predict and evaluate on real test data
y_pred = model.predict(X_test_real)
accuracy = accuracy_score(y_test_real, y_pred)
print(f"Model Accuracy on Test Set (trained on fake, tested on real): {accuracy * 100:.2f}%")
print("Confusion Matrix:")
print(confusion_matrix(y_test_real, y_pred))
print("\nClassification Report:")
print(classification_report(y_test_real, y_pred, zero_division=0))

Model Accuracy on Test Set (trained on fake, tested on real): 83.23%
Confusion Matrix:
[[2130  370]
 [ 116  282]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.85      0.90      2500
           1       0.43      0.71      0.54       398

    accuracy                           0.83      2898
   macro avg       0.69      0.78      0.72      2898
weighted avg       0.88      0.83      0.85      2898

